# XGBoost Models: Infertility, Dysmenorrhea, PMDD, T2D, CVD, Endometrial

For each disease this notebook trains:
- an **XGBClassifier** (target: `<Disease>_ActualFlag`) with severity categories from predicted probability
- an **XGBRegressor** (target: `<Disease>_ActualScore`, risk score 0-1) with severity categories from the score

Per-patient predictions are **out-of-fold** (cross-validated). Final models are refit on all data and saved as `.pkl` and `.joblib`.

> **Caveat:** the input file holds outputs of an earlier model stage (`PredFlag`, `PredScore`, `RiskProb`), not raw clinical data, so the metrics are inflated.

In [1]:
import os, warnings, pickle, joblib
import numpy as np, pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score,
                             r2_score, mean_absolute_error, mean_squared_error)
from openpyxl.styles import Font
warnings.filterwarnings("ignore")

DATA = "/mnt/user-data/uploads/symptom_intensity_predictions_full.csv"
OUT = "/mnt/user-data/outputs"
os.makedirs(f"{OUT}/models", exist_ok=True)
SEED = 42

## 1. Load data

In [2]:
df = pd.read_csv(DATA)
DISEASES = ["Infertility", "Dysmenorrhea", "PMDD", "T2D", "CVD", "Endometrial"]
print(df.shape, "| missing values:", int(df.isna().sum().sum()))
df.head()

(1082, 32) | missing values: 0


,Patient_File_No,pcos_label,Infertility_ActualFlag,Infertility_ActualScore,Infertility_PredFlag,Infertility_PredScore,Infertility_RiskProb,Dysmenorrhea_ActualFlag,Dysmenorrhea_ActualScore,Dysmenorrhea_PredFlag,...,CVD_ActualFlag,CVD_ActualScore,CVD_PredFlag,CVD_PredScore,CVD_RiskProb,Endometrial_ActualFlag,Endometrial_ActualScore,Endometrial_PredFlag,Endometrial_PredScore,Endometrial_RiskProb
0,1,1,1,0.6060,1,0.6458,0.9946,0,0.0000,1,...,0,0.2565,0,0.2548,0.0034,0,0.2808,0,0.2828,0.0000
1,2,0,0,0.0946,0,0.0959,0.0061,1,0.5086,1,...,0,0.1777,0,0.1743,0.0008,0,0.1997,0,0.1996,0.0001
2,3,1,1,0.6648,1,0.6657,0.9946,0,0.3174,0,...,0,0.3377,0,0.3384,0.0050,0,0.2740,0,0.2728,0.0000
3,4,1,1,0.6235,1,0.6301,0.9946,1,0.4693,1,...,0,0.3366,0,0.3361,0.0009,0,0.2708,0,0.2733,0.0001
4,5,0,0,0.1188,0,0.1194,0.0061,1,0.5018,1,...,0,0.2249,0,0.2268,0.0022,0,0.1781,0,0.1768,0.0000


## 2. Features, targets and severity bands

- **Features:** `pcos_label` plus `PredFlag`, `PredScore`, `RiskProb` for all six diseases. `Actual*` columns are never features.
- **Classification severity:** Low < 0.33, Moderate 0.33-0.66, High >= 0.66
- **Regression severity:** Minimal < 0.2, Mild 0.2-0.4, Moderate 0.4-0.6, Severe >= 0.6

In [3]:
FEATURES = ["pcos_label"] + [f"{d}_{s}" for d in DISEASES for s in ("PredFlag", "PredScore", "RiskProb")]
X = df[FEATURES]

def clf_category(p):
    return pd.cut(p, [-np.inf, .33, .66, np.inf], labels=["Low", "Moderate", "High"]).astype(str)

def reg_category(s):
    return pd.cut(s, [-np.inf, .2, .4, .6, np.inf], labels=["Minimal", "Mild", "Moderate", "Severe"]).astype(str)

pd.DataFrame({"Disease": DISEASES,
              "N_positive": [int(df[f"{d}_ActualFlag"].sum()) for d in DISEASES],
              "Prevalence": [round(df[f"{d}_ActualFlag"].mean(), 4) for d in DISEASES]})

,Disease,N_positive,Prevalence
0,Infertility,508,0.4695
1,Dysmenorrhea,730,0.6747
2,PMDD,447,0.4131
3,T2D,192,0.1774
4,CVD,3,0.0028
5,Endometrial,11,0.0102


## 3. Train, cross-validate and save
Cross-validation folds are capped at the number of positives (CVD has only 3). Classifiers use `scale_pos_weight` for imbalance.

In [4]:
clf_table = pd.DataFrame({"Patient_File_No": df.Patient_File_No, "pcos_label": df.pcos_label})
reg_table = clf_table.copy()
metrics = []
bundle = {"features": FEATURES, "classifiers": {}, "regressors": {},
          "clf_bins": "Low<0.33<=Moderate<0.66<=High",
          "reg_bins": "Minimal<0.2<=Mild<0.4<=Moderate<0.6<=Severe"}

for d in DISEASES:
    y_c = df[f"{d}_ActualFlag"]
    y_r = df[f"{d}_ActualScore"].clip(0, 1)
    n_pos = int(y_c.sum())
    k = max(2, min(5, n_pos))
    spw = (len(y_c) - n_pos) / max(n_pos, 1)

    clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                        scale_pos_weight=spw, eval_metric="logloss", random_state=SEED, n_jobs=-1)
    reg = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                       objective="reg:squarederror", random_state=SEED, n_jobs=-1)

    # out-of-fold predictions
    p = cross_val_predict(clf, X, y_c, cv=StratifiedKFold(k, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    s = np.clip(cross_val_predict(reg, X, y_r, cv=KFold(k, shuffle=True, random_state=SEED)), 0, 1)
    pred = (p >= .5).astype(int)

    clf_table[f"{d}_Actual"] = y_c.values
    clf_table[f"{d}_Prob"] = p.round(4)
    clf_table[f"{d}_Pred"] = pred
    clf_table[f"{d}_Severity"] = clf_category(p)

    reg_table[f"{d}_ActualScore"] = y_r.values.round(4)
    reg_table[f"{d}_RiskScore"] = s.round(4)
    reg_table[f"{d}_Severity"] = reg_category(s)

    metrics.append({"Disease": d, "N_positive": n_pos, "CV_folds": k,
                    "AUC": roc_auc_score(y_c, p), "PR_AUC": average_precision_score(y_c, p),
                    "F1": f1_score(y_c, pred, zero_division=0), "BalancedAcc": balanced_accuracy_score(y_c, pred),
                    "Reg_R2": r2_score(y_r, s), "Reg_MAE": mean_absolute_error(y_r, s),
                    "Reg_RMSE": mean_squared_error(y_r, s) ** .5})

    # final models on all data
    clf.fit(X, y_c); reg.fit(X, y_r)
    bundle["classifiers"][d] = clf
    bundle["regressors"][d] = reg
    for kind, m in (("classifier", clf), ("regressor", reg)):
        with open(f"{OUT}/models/{d}_{kind}.pkl", "wb") as f:
            pickle.dump(m, f)
        joblib.dump(m, f"{OUT}/models/{d}_{kind}.joblib")
    print(f"done: {d}")

with open(f"{OUT}/models/all_models_bundle.pkl", "wb") as f:
    pickle.dump(bundle, f)
joblib.dump(bundle, f"{OUT}/models/all_models_bundle.joblib")

done: Infertility


done: Dysmenorrhea


done: PMDD


done: T2D


done: CVD


done: Endometrial


['/mnt/user-data/outputs/models/all_models_bundle.joblib']

## 4. Model metrics (cross-validated)

In [5]:
met = pd.DataFrame(metrics).round(4)
met

,Disease,N_positive,CV_folds,AUC,PR_AUC,F1,BalancedAcc,Reg_R2,Reg_MAE,Reg_RMSE
0,Infertility,508,5,1.0000,1.0000,1.0000,1.0000,0.9991,0.0037,0.0078
1,Dysmenorrhea,730,5,0.9987,0.9994,0.9849,0.9790,0.9758,0.0090,0.0137
2,PMDD,447,5,1.0000,1.0000,1.0000,1.0000,0.9930,0.0052,0.0090
3,T2D,192,5,1.0000,1.0000,0.9974,0.9974,0.9964,0.0034,0.0066
4,CVD,3,3,1.0000,1.0000,1.0000,1.0000,0.9899,0.0035,0.0069
5,Endometrial,11,5,1.0000,1.0000,0.9524,0.9545,0.9976,0.0021,0.0028


## 5. Per-patient table: Classification (XGBClassifier)

In [6]:
clf_cols = ["Patient_File_No", "pcos_label"] + [f"{d}_{s}" for d in DISEASES for s in ("Prob", "Severity")]
clf_table[clf_cols].head(20)

,Patient_File_No,pcos_label,Infertility_Prob,Infertility_Severity,Dysmenorrhea_Prob,Dysmenorrhea_Severity,PMDD_Prob,PMDD_Severity,T2D_Prob,T2D_Severity,CVD_Prob,CVD_Severity,Endometrial_Prob,Endometrial_Severity
0,1,1,0.9974,High,0.9654,High,0.0023,Low,0.0015,Low,0.0016,Low,0.0016,Low
1,2,0,0.0026,Low,0.9984,High,0.9977,High,0.0015,Low,0.0016,Low,0.0011,Low
2,3,1,0.9974,High,0.0013,Low,0.0023,Low,0.9984,High,0.0017,Low,0.0017,Low
3,4,1,0.9974,High,0.9974,High,0.9977,High,0.9984,High,0.0018,Low,0.0015,Low
4,5,0,0.0026,Low,0.9884,High,0.9977,High,0.0015,Low,0.0016,Low,0.0011,Low
5,6,1,0.9974,High,0.9119,High,0.0022,Low,0.0015,Low,0.0015,Low,0.0011,Low
6,7,1,0.9974,High,0.9990,High,0.9977,High,0.9984,High,0.0017,Low,0.0017,Low
7,8,1,0.9974,High,0.9987,High,0.9977,High,0.0015,Low,0.0014,Low,0.0012,Low
8,9,0,0.0026,Low,0.9968,High,0.9977,High,0.0015,Low,0.0014,Low,0.0011,Low
9,10,1,0.9974,High,0.9436,High,0.0023,Low,0.9983,High,0.0014,Low,0.0014,Low


## 6. Per-patient table: Regression (XGBRegressor, risk score 0-1)

In [7]:
reg_cols = ["Patient_File_No", "pcos_label"] + [f"{d}_{s}" for d in DISEASES for s in ("RiskScore", "Severity")]
reg_table[reg_cols].head(20)

,Patient_File_No,pcos_label,Infertility_RiskScore,Infertility_Severity,Dysmenorrhea_RiskScore,Dysmenorrhea_Severity,PMDD_RiskScore,PMDD_Severity,T2D_RiskScore,T2D_Severity,CVD_RiskScore,CVD_Severity,Endometrial_RiskScore,Endometrial_Severity
0,1,1,0.6474,Severe,0.2620,Mild,0.1834,Minimal,0.3515,Mild,0.2558,Mild,0.2804,Mild
1,2,0,0.0964,Minimal,0.5069,Moderate,0.4954,Moderate,0.1671,Minimal,0.1736,Minimal,0.1991,Minimal
2,3,1,0.6711,Severe,0.2964,Mild,0.3833,Mild,0.4449,Moderate,0.3418,Mild,0.2688,Mild
3,4,1,0.6318,Severe,0.4725,Moderate,0.5177,Moderate,0.4243,Moderate,0.3351,Mild,0.2725,Mild
4,5,0,0.1197,Minimal,0.4840,Moderate,0.4904,Moderate,0.2390,Mild,0.2240,Mild,0.1777,Minimal
5,6,1,0.6224,Severe,0.3626,Mild,0.3794,Mild,0.3631,Mild,0.2831,Mild,0.2828,Mild
6,7,1,0.6872,Severe,0.4490,Moderate,0.4730,Moderate,0.4176,Moderate,0.3044,Mild,0.2894,Mild
7,8,1,0.6598,Severe,0.4395,Moderate,0.4422,Moderate,0.3567,Mild,0.2740,Mild,0.3050,Mild
8,9,0,0.1173,Minimal,0.4356,Moderate,0.4529,Moderate,0.1503,Minimal,0.1562,Minimal,0.1622,Minimal
9,10,1,0.6117,Severe,0.3526,Mild,0.3423,Mild,0.4028,Moderate,0.3117,Mild,0.3157,Mild


## 7. Severity distribution

In [8]:
dist = []
for d in DISEASES:
    for name, tbl in (("Classification", clf_table), ("Regression", reg_table)):
        vc = tbl[f"{d}_Severity"].value_counts()
        dist.append({"Disease": d, "Model": name, **vc.to_dict()})
pd.DataFrame(dist).fillna(0).set_index(["Disease", "Model"]).astype(int)

Low  High  Minimal  Severe  Moderate  Mild
Disease      Model                                                      
Infertility  Classification   574   508        0       0         0     0
             Regression         0     0      562     378       130    12
Dysmenorrhea Classification   352   721        0       0         9     0
             Regression         0     0        1       6       543   532
PMDD         Classification   635   447        0       0         0     0
             Regression         0     0       15      17       505   545
T2D          Classification   891   191        0       0         0     0
             Regression         0     0      362       0       193   527
CVD          Classification  1079     3        0       0         0     0
             Regression         0     0      385       0         0   697
Endometrial  Classification  1072    10        0       0         0     0
             Regression         0     0      310       0         0   772

## 8. Export tables

In [9]:
clf_table.to_csv(f"{OUT}/classification_predictions_per_patient.csv", index=False)
reg_table.to_csv(f"{OUT}/regression_predictions_per_patient.csv", index=False)
met.to_csv(f"{OUT}/model_metrics.csv", index=False)

with pd.ExcelWriter(f"{OUT}/per_patient_predictions.xlsx", engine="openpyxl") as w:
    clf_table.to_excel(w, sheet_name="Classification", index=False)
    reg_table.to_excel(w, sheet_name="Regression", index=False)
    met.to_excel(w, sheet_name="Metrics", index=False)
    for ws in w.book.worksheets:
        ws.freeze_panes = "C2"
        for c in ws[1]:
            c.font = Font(bold=True, name="Arial")
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = max(12, len(str(col[0].value)) + 2)
print("exported")

exported


## 9. Reload check and inference example

In [10]:
b_job = joblib.load(f"{OUT}/models/all_models_bundle.joblib")
with open(f"{OUT}/models/all_models_bundle.pkl", "rb") as f:
    b_pkl = pickle.load(f)

sample = df[b_job["features"]].head(3)
for d in DISEASES:
    p_j = b_job["classifiers"][d].predict_proba(sample)[:, 1]
    p_p = b_pkl["classifiers"][d].predict_proba(sample)[:, 1]
    assert np.allclose(p_j, p_p), d
    s = b_job["regressors"][d].predict(sample).clip(0, 1)
    print(d, "prob:", p_j.round(3), "| risk score:", s.round(3))
print("pickle and joblib bundles give identical predictions")

Infertility prob: [0.998 0.002 0.998] | risk score: [0.613 0.095 0.666]
Dysmenorrhea prob: [0.19  0.999 0.001] | risk score: [0.012 0.512 0.308]
PMDD prob: [0.002 0.998 0.002] | risk score: [0.009 0.498 0.379]
T2D prob: [0.001 0.001 0.999] | risk score: [0.353 0.169 0.449]
CVD prob: [0.001 0.001 0.002] | risk score: [0.257 0.176 0.341]
Endometrial prob: [0.001 0.001 0.001] | risk score: [0.281 0.199 0.271]
pickle and joblib bundles give identical predictions
